# Train small-lm on Colab

Runs `data.py` / `train.py` / `plot_loss.py` from the repo unmodified. The
actual run behind this repo's README was trained on **Colab Pro (a single
L4)**; the notebook works unmodified on the free T4 tier too, just slower.

**Before you start:** Runtime menu -> Change runtime type -> pick your GPU
(**L4** on Pro, **T4** on the free tier).

**Realities this notebook works around, Pro or free:**
- Colab's local disk is wiped every time the session disconnects/recycles (idle timeout, or the runtime's hard cap). So data and checkpoints live on **Google Drive** instead, and training runs with `--resume` so a disconnect costs you nothing but time.
- GPUs can still get pre-empted under load, even on Pro. Just re-run the last two cells when that happens -- `--resume` picks up from `out/ckpt.pt`.
- Keep the browser tab open/active; idle detection disconnects background tabs.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Everything persistent (data + checkpoints) lives under this Drive folder
# so it survives a disconnect. Only the repo code is re-cloned each session.
PROJECT_DIR = '/content/drive/MyDrive/small-lm'
import os
os.makedirs(PROJECT_DIR, exist_ok=True)
print('persistent project dir:', PROJECT_DIR)

## Get the code

Replace the URL with your GitHub repo once you've pushed it. If you haven't pushed yet,
use the fallback cell below to upload a zip instead.

In [ ]:
REPO_URL = 'https://github.com/eric-solak/musicGPT.git'

%cd /content
!git clone $REPO_URL musicGPT
%cd musicGPT

In [ ]:
# Build the dataset directly onto Drive, so it only needs downloading once ever.
DATA_DIR = f'{PROJECT_DIR}/data'

import os
if os.path.exists(f'{DATA_DIR}/train.bin'):
    print('data already built, skipping')
else:
    !python data.py --source enwiki --parts 4 --data-dir "$DATA_DIR"

## Train

Run this cell. If the session disconnects, just re-run it -- `--resume` continues from
the last checkpoint on Drive instead of starting over.

In [ ]:
OUT_DIR = f'{PROJECT_DIR}/out'

# --max-steps 14000: 14k steps * 65,536 tokens/step ~= 0.9B tokens, the
# Chinchilla-ish ~20 tokens/param budget for the 45M preset. Passing it on
# resume reshapes the cosine so the LR anneals to min_lr by step 14k instead
# of stopping stranded partway down the original 20k-step schedule.
# --eval-interval 100: each eval runs 100 val batches plus sample generation,
# so evaluating every 50 steps spends real time re-checking; 100 halves that
# overhead while still capping preemption loss at ~100 steps.
resume_flag = '--resume' if os.path.exists(f'{OUT_DIR}/ckpt.pt') else ''
!python train.py --model base --data-dir "$DATA_DIR" --out-dir "$OUT_DIR" \
    --batch-size 16 --grad-accum-steps 8 --eval-interval 100 \
    --max-steps 14000 {resume_flag}

## Check progress

Safe to run any time, including from a second Colab tab while training runs in the first.

In [ ]:
!python plot_loss.py --out-dir "$OUT_DIR"
from IPython.display import Image, display
display(Image(f'{OUT_DIR}/loss_curve.png'))

In [ ]:
!python sample.py --ckpt "$OUT_DIR/ckpt_best.pt" --prompt "The electric guitar" --num-samples 2

## When training is done

Everything you need is on Drive at `PROJECT_DIR`:
- `out/ckpt_best.pt` -- the best-val-loss checkpoint, for sampling and any future demo/fine-tune
- `out/metrics.csv`, `out/loss_curve.png` -- for the README's Results section
- `out/samples/step_*.txt` -- generations over time, for the before/after in the README

Download `metrics.csv`, `loss_curve.png`, and a couple of `samples/*.txt` from Drive
into the local repo's `out/` folder and commit them (.gitignore already allows exactly
these). Keep `ckpt_best.pt` on Drive; it is too big to commit.